[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C19_Bayesian_ML_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零实现**贝叶斯推断算法，再与**解析解 / 暴力枚举**等可信参照 **对拍**。

这个 notebook 做四件事：① 确认环境；② 用一个最小例子体会**贝叶斯更新**（先验→后验）；③ 立下全课的纪律——**对拍（differential testing）**；④ 演示**对数空间**计算为何是贝叶斯数值的命根子。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画后验/采样轨迹）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 最小的贝叶斯更新：一枚硬币

抛一枚硬币，正面概率 $\theta$ 未知。先验设为 $\mathrm{Beta}(2,2)$（温和地相信接近公平）。观测 $n=10$ 次中 $k=8$ 次正面。**Beta-Binomial 共轭**给出闭式后验：

$$ p(\theta\mid D) = \mathrm{Beta}(\alpha+k,\; \beta+n-k) = \mathrm{Beta}(2+8,\; 2+2) = \mathrm{Beta}(10, 4) $$

更新规则朴素到惊人：**把成功数加到 $\alpha$、失败数加到 $\beta$**。下面我们不直接用公式，而是用**网格数值积分**独立算一遍后验，再和共轭公式对拍——这就是全课的工作流。

In [ ]:
# 先验 Beta(a0,b0)，似然 Binomial(n,k)，目标后验
from math import lgamma
# 版本无关的梯形积分（numpy<2 用 trapz，numpy>=2 用 trapezoid）
trapz = getattr(np, 'trapezoid', None) or np.trapz
a0, b0 = 2.0, 2.0
n, k = 10, 8

def beta_logpdf(x, a, b):
    # log Beta(x; a,b) = (a-1)log x + (b-1)log(1-x) - logB(a,b)
    logB = lgamma(a) + lgamma(b) - lgamma(a + b)
    return (a - 1) * np.log(x) + (b - 1) * np.log1p(-x) - logB

# 共轭闭式后验
a_post, b_post = a0 + k, b0 + (n - k)
print(f'共轭后验: Beta({a_post}, {b_post})')

# 网格数值积分：后验 ∝ 似然 × 先验，逐点算再归一化
grid = np.linspace(1e-6, 1 - 1e-6, 20001)
log_prior = beta_logpdf(grid, a0, b0)
log_like = k * np.log(grid) + (n - k) * np.log1p(-grid)   # 略去与 θ 无关的组合数
log_unnorm = log_prior + log_like
# 归一化（在网格上做梯形积分）
unnorm = np.exp(log_unnorm - log_unnorm.max())
Z = trapz(unnorm, grid)
post_numeric = unnorm / Z

# 对拍：网格后验 vs 共轭闭式后验
post_closed = np.exp(beta_logpdf(grid, a_post, b_post))
max_err = np.max(np.abs(post_numeric - post_closed))
print(f'网格后验 vs 共轭闭式后验  max|err| = {max_err:.2e}')
assert max_err < 1e-3, '数值后验应与共轭闭式后验一致'
print('✅ 对拍通过：贝叶斯更新 = 似然 × 先验再归一化，共轭只是它的闭式捷径')

## 3 · 后验告诉我们什么：点估计 vs 整个分布

频率派给一个数（MLE $=k/n=0.8$）。贝叶斯给**整个后验** $\mathrm{Beta}(10,4)$，从中可读出：
- **后验均值** $\frac{a}{a+b}$（被先验向 0.5 收缩，故小于 MLE 的 0.8）；
- **后验众数（MAP）** $\frac{a-1}{a+b-2}$；
- **95% 可信区间**：参数有 95% 概率落在其中——这是置信区间**不能**说的。

In [ ]:
a, b = a_post, b_post
mle = k / n
post_mean = a / (a + b)
post_mode = (a - 1) / (a + b - 2)
post_var = a * b / ((a + b) ** 2 * (a + b + 1))
print(f'MLE (频率派点估计)     = {mle:.4f}')
print(f'后验均值 (贝叶斯)       = {post_mean:.4f}  <- 被先验收缩，< MLE')
print(f'后验众数 MAP           = {post_mode:.4f}')
print(f'后验标准差 (不确定性)   = {post_var**0.5:.4f}')

# 95% 等尾可信区间：用网格后验的 CDF 反查分位
cdf = np.cumsum(post_closed) * (grid[1] - grid[0])
cdf /= cdf[-1]
lo = grid[np.searchsorted(cdf, 0.025)]
hi = grid[np.searchsorted(cdf, 0.975)]
print(f'95% 可信区间           = [{lo:.3f}, {hi:.3f}]')
assert post_mean < mle, '先验 Beta(2,2) 应把后验均值从 MLE 向 0.5 收缩'
assert lo < post_mean < hi
print('✅ 后验不是一个数，而是一整套关于 θ 的信念 —— 这是贝叶斯的核心产物')

## 4 · 立纪律：对拍（differential testing）

本课每个算法都要和一个**绝对可信的参照**比对。把它封装成一个小工具，后面每个模块都用它判定「我的实现 == 参照」。它就是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_allclose(name, got, ref, atol=1e-8, rtol=1e-5):
    '''对拍：被测实现结果 vs 可信参照。返回是否一致并打印 max|err|。'''
    got = np.asarray(got, dtype=float)
    ref = np.asarray(ref, dtype=float)
    ok = np.allclose(got, ref, atol=atol, rtol=rtol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参照不一致！'
    return ok

# 演示：用蒙特卡洛估计 Beta 后验均值，对拍解析均值
rng = np.random.default_rng(0)
samples = rng.beta(a_post, b_post, size=200000)
check_allclose('MC mean vs 解析均值', samples.mean(), a_post / (a_post + b_post), atol=2e-3)
print('\n这就是全课的工作流：写实现 -> 对拍可信参照 -> assert 兜底。')

## 5 · 对数空间：贝叶斯数值的命根子

似然是大量小概率连乘，直接相乘会**下溢为 0**；后验密度的指数项会**上溢为 inf**。贝叶斯计算几乎永远在**对数空间**进行：连乘变连加，再用 **log-sum-exp** 技巧稳定地做归一化/求和。

$$ \log\sum_i e^{x_i} = m + \log\sum_i e^{x_i - m}, \qquad m=\max_i x_i $$

In [ ]:
def logsumexp(x):
    x = np.asarray(x, dtype=float)
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

# 演示：1000 个样本的似然连乘
p = rng.uniform(0.001, 0.05, size=1000)        # 每个都是小概率
naive = np.prod(p)                              # 朴素连乘
log_correct = np.sum(np.log(p))                 # 对数空间连加
print(f'朴素连乘 prod(p)        = {naive}        <- 下溢为 0')
print(f'对数空间 sum(log p)     = {log_correct:.2f}  (= log 真实值)')
assert naive == 0.0, '朴素连乘应下溢为 0'
assert np.isfinite(log_correct), '对数空间应保持有限'

# log-sum-exp 对拍朴素 log(sum(exp)))（小数值时两者应一致）
x = np.array([-1.0, -2.0, -3.0])
check_allclose('logsumexp vs naive', logsumexp(x), np.log(np.sum(np.exp(x))), atol=1e-12)
print('✅ 对数空间 + log-sum-exp：贝叶斯一切归一化/求和的数值地基（模块 02/03/05 反复用）')

## 6 · 蒙特卡洛：高维积分的钥匙

贝叶斯里到处是积分（后验预测、证据、期望）。**蒙特卡洛**用样本均值近似期望：

$$ \mathbb{E}_{p}[f(x)] \approx \frac{1}{N}\sum_{i=1}^N f(x_i),\quad x_i\sim p $$

它**无偏**、误差按 $O(1/\sqrt N)$ 下降、**与维度无关**——这正是 MCMC（模块 02）能攻克高维后验积分的根本原因。

In [ ]:
# 用蒙特卡洛估计 Beta(10,4) 下 E[θ] 与 E[θ^2]，对拍解析值
a, b = 10.0, 4.0
N = 500000
s = rng.beta(a, b, size=N)
mc_mean = s.mean()
mc_m2 = (s ** 2).mean()
ana_mean = a / (a + b)
ana_m2 = a * (a + 1) / ((a + b) * (a + b + 1))
print(f'E[θ]   MC={mc_mean:.4f}  解析={ana_mean:.4f}')
print(f'E[θ^2] MC={mc_m2:.4f}  解析={ana_m2:.4f}')
check_allclose('MC E[θ]',   mc_mean, ana_mean, atol=3e-3)
check_allclose('MC E[θ^2]', mc_m2,   ana_m2,   atol=3e-3)

# 误差随 N 的 1/sqrt(N) 衰减：N 翻 100 倍，误差约降 10 倍
err_small = abs(rng.beta(a, b, size=1000).mean() - ana_mean)
err_large = abs(rng.beta(a, b, size=100000).mean() - ana_mean)
print(f'\nN=1e3 误差≈{err_small:.4f}, N=1e5 误差≈{err_large:.4f} (约降 ~10x)')
print('✅ 蒙特卡洛：采样换积分，误差 O(1/√N) 与维度无关 —— MCMC 的理论靠山')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个共轭更新/采样器/变分优化/GP/图推断，都会用 `check_allclose` 对拍一个可信参照（解析后验、真分布的矩、暴力枚举的边缘）；结构正确则数值一致，数值一致则逻辑可迁移到 PyMC/Stan/GPyTorch。

**接下来五个内容模块**：01 贝叶斯推断与共轭 → 02 MCMC → 03 变分推断 → 04 高斯过程 → 05 概率图模型。

下一站：**模块 01 · 贝叶斯推断与共轭**。